# Caracteristicas del texto (TF-IDF)

Notebook de la etapa de caracteristicas del pipeline. Aca se explora como representar el texto
preprocesado de la etapa 00 con TF-IDF y se comparan dos configuraciones del `TfidfVectorizer`.

> **Nota**: este analisis es exploratorio. La configuracion final la define la etapa de modelado.


## Importacion de librerias

Se importan las librerias de manipulacion de datos (`pandas`, `numpy`) y la de vectorizacion
(`TfidfVectorizer`), que implementa la representacion TF-IDF.


In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer


## Vectorizacion TF-IDF

**TF-IDF** (*Term Frequency - Inverse Document Frequency*, frecuencia de termino - frecuencia
inversa de documento) es una representacion numerica del texto: convierte cada documento en un
vector donde cada componente es una **frecuencia ponderada** de un termino del vocabulario.

### ¿Que es una frecuencia ponderada?

Una frecuencia simple, como la del BoW, cuenta cuantas veces aparece cada palabra en un texto.
El problema es que las palabras mas comunes del corpus (`school`, `like`, `people`) aparecen en
casi todos los documentos y aportan poca informacion para distinguir ciberacoso de no ciberacoso.
La frecuencia ponderada corrige esto penalizando los terminos que aparecen en muchos documentos.

El peso TF-IDF de un termino $t$ en un documento $d$ es el producto de dos factores:

$$ \mathrm{tfidf}(t,d) = \mathrm{tf}(t,d) \times \mathrm{idf}(t) $$

- **TF** es la frecuencia de termino: cuantas veces aparece $t$ en $d$.
- **IDF** es la frecuencia inversa de documento: mide cuan raro es $t$ en todo el corpus.
  Penaliza las palabras que aparecen en casi todos los documentos. Con el suavizado por defecto
  de scikit-learn:

$$ \mathrm{idf}(t) = \ln\left(\frac{1 + N}{1 + \mathrm{df}(t)}\right) + 1 $$

donde $N$ es el numero total de documentos y $\mathrm{df}(t)$ cuantos documentos contienen a $t$.
Cuanto mas comun es $t$, mas se acerca $\mathrm{idf}(t)$ a 1. Cuanto mas raro, mas crece.

Resultado: un termino recibe peso alto solo si es **frecuente en el documento** y **raro en el
corpus**. Es lo contrario de las palabras comunes, que no discriminan.


## Carga de datos

Se lee el CSV con el texto ya preprocesado en la etapa 00.


In [2]:
df = pd.read_csv('../data/processed/cyberbullying_preprocessed.csv')
print('Filas:', len(df))
df[['text', 'text_preprocessed', 'label']].head()


Filas: 80974


,text,text_preprocessed,label
0,"In other words #katandandre, your food was cra...",word food crapilicious,0
1,Why is #aussietv so white? #MKR #theblock #ImA...,white,0
2,@XochitlSuckkks a classy whore? Or more red ve...,classy whore red velvet cupcake,0
3,"@Jason_Gio meh. :P thanks for the heads up, b...",meh thank head not_concerned angry dude twitter,0
4,@RudhoeEnglish This is an ISIS account pretend...,isis account pretend kurdish account like isla...,0


## Dos configuraciones: con filtros y sin filtros

Para decidir como representar el texto, se comparan dos configuraciones del `TfidfVectorizer`.

La primera filtra el vocabulario. Descarta todo termino que aparece en menos del 1 % de los
documentos (`min_df=0.01`) y limita el vocabulario a 1000 terminos como maximo
(`max_features=1000`).

La segunda usa los valores por defecto, que no filtran nada. Cada palabra del corpus entra en el
vocabulario, aparezca una vez o mil. Es la configuracion que usa el pipeline de la etapa de
modelado.


In [3]:
# Configuracion filtrada: descarta terminos raros y limita el vocabulario.
tfidf_filtered = TfidfVectorizer(min_df=0.01, max_features=1000)
X_filtered = tfidf_filtered.fit_transform(df['text_preprocessed'])

# Configuracion completa: sin filtros, la que usa el pipeline de modelado.
tfidf_full = TfidfVectorizer()
X_full = tfidf_full.fit_transform(df['text_preprocessed'])

print('Con filtros (min_df=0.01, max_features=1000):', X_filtered.shape)
print('Sin filtros:', X_full.shape)
print('Reduccion del vocabulario con filtros: {:.1f} %'.format(
    (1 - X_filtered.shape[1] / X_full.shape[1]) * 100
))


Con filtros (min_df=0.01, max_features=1000): (80974, 118)
Sin filtros: (80974, 38709)
Reduccion del vocabulario con filtros: 99.7 %


### Interpretacion: el filtro deja muy poco vocabulario

Con filtros, la matriz queda de **80.974 documentos por 118 terminos**. De las 38.709 palabras
distintas del corpus, solo 118 sobreviven. El filtro descarta todo lo que aparece en menos del
1 % de los documentos, y como los mensajes son cortos, casi todas las palabras quedan afuera.

Sin filtros, la matriz es de **80.974 documentos por 38.709 terminos**. El vocabulario completo
entra en la representacion. Cada mensaje se convierte en un vector con un peso TF-IDF por cada
palabra del vocabulario.

La conclusion de la comparacion es directa: filtrar con un umbral del 1 % deja afuera casi todo
el vocabulario. Esa reduccion es tan agresiva que elimina palabras que podrian servir para
clasificar. Por eso el pipeline de modelado no filtra.

> **Nota**: la cifra de 38.709 corresponde a ajustar el vectorizador sobre el corpus completo.
> En la etapa de modelado el vectorizador se ajusta sobre el split de entrenamiento (80 %), donde
> el vocabulario es de **34.855 terminos**. Ese es el tamaño real del vocabulario que ve el modelo.


## Pesos IDF del modelo entrenado

La cifra de 38.709 terminos corresponde al corpus completo. Para igualar la configuracion de
produccion, se replica la particion de la etapa de modelado: 80 % para entrenar, 20 % para
evaluar, con `random_state=1` y estratificada por clase. Sobre el split de entrenamiento se
ajusta un `TfidfVectorizer` sin filtros, igual que en la etapa de modelado.


In [4]:
from sklearn.model_selection import train_test_split

# Misma particion que la etapa de modelado (80/20, estratificada).
X_train, X_test, y_train, y_test = train_test_split(
    df['text_preprocessed'], df['label'],
    test_size=0.2, random_state=1, stratify=df['label'],
)

# Vectorizador sin filtros ajustado solo sobre el entrenamiento, como en produccion.
tfidf_model = TfidfVectorizer()
X_model = tfidf_model.fit_transform(X_train)

print('Split de entrenamiento:', X_train.shape[0], 'mensajes')
print('Vocabulario del modelo:', X_model.shape[1], 'terminos')


Split de entrenamiento: 64779 mensajes
Vocabulario del modelo: 34855 terminos


Se calcula la metrica IDF para cada palabra del vocabulario del modelo y se ordena de mayor a
menor.


In [5]:
features = tfidf_model.get_feature_names_out()
df_idf = (
    pd.DataFrame({'word': features, 'idf': tfidf_model.idf_})
    .sort_values('idf', ascending=False)
    .reset_index(drop=True)
)
df_idf.head(10)


,word,idf
0,aab,11.385605
1,nay,11.385605
2,nby,11.385605
3,nbody,11.385605
4,nazis,11.385605
5,naziophobe,11.385605
6,naziism,11.385605
7,naziarmy,11.385605
8,nazareth,11.385605
9,nayef,11.385605


In [6]:
df_idf.tail(10)


,word,idf
34845,dumb,3.832318
34846,ass,3.804650
34847,high,3.758061
34848,hate,3.700591
34849,girl,3.695090
34850,people,3.685989
34851,fuck,3.389456
34852,like,3.377905
34853,school,3.294590
34854,bully,3.210057


### Interpretacion: pesos IDF

El DataFrame muestra los pesos IDF del vocabulario del modelo, ordenados de mayor a menor. Un
IDF alto significa que el termino es **poco frecuente en el corpus** y, por lo tanto, gana mas
peso relativo cuando aparece en un documento. Un IDF cercano a 1 corresponde a los terminos mas
comunes, que son los menos informativos.

Los valores mas altos (11,39) son terminos que aparecen en muy pocos mensajes de entrenamiento.
Varios son ofensivos, como `nazis`, `naziism` o `naziophobe`. Si aparecen en un mensaje, el
modelo les da un peso muy alto.

Los valores mas bajos son los terminos comunes de todo el corpus. `bully` (3,21), `school` (3,29)
o `fuck` (3,39) aparecen en casi todos los documentos, asi que no ayudan a distinguir una clase
de la otra. Hasta la palabra `bully` tiene poco peso, porque no es exclusiva de los mensajes de
ciberacoso.

Esto conecta con el hallazgo del EDA: si un termino raro pero muy asociado a la clase de
ciberacoso aparece en un texto, el modelo le otorga un peso desproporcionado. Eso refuerza el
riesgo de falsos positivos sobre los terminos de identidad.


## Resumen del analisis de caracteristicas

En esta etapa se compararon dos configuraciones de **TF-IDF**:

- Con filtros (`min_df=0.01`, `max_features=1000`): vocabulario de 118 terminos.
- Sin filtros (valores por defecto): 38.709 terminos sobre el corpus completo, y 34.855 sobre el
  split de entrenamiento. Esa ultima es la configuracion de la etapa de modelado.

Se inspeccionaron:

- Las **matrices TF-IDF** de ambas configuraciones.
- Los **pesos IDF** del vocabulario del modelo.


### Conclusion de la etapa

Filtrar con un umbral del 1 % deja solo 118 terminos y descarta casi todo el vocabulario. Esa
reduccion es tan agresiva que elimina palabras que podrian servir para clasificar. La
configuracion sin filtros conserva el vocabulario completo y es la que usa la etapa de modelado,
que al ajustarse sobre el entrenamiento termina con **34.855 terminos**.

TF-IDF tiene dos limitaciones que se tienen en cuenta al interpretar los resultados. No considera
el orden de las palabras dentro del mensaje. Y los filtros de frecuencia, cuando se usan con
umbrales altos, pueden dejar afuera justo el vocabulario discriminante. Ambas se retoman en la
discusion de resultados de la etapa de modelado.
